In [15]:
import logging
import time
import sys

In [16]:
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    stream=sys.stdout,
                    force=True
                    )

In [17]:
def extract():
    logging.info("Extract step")
    return [
        {"user": "Ravi", "amount": 1200},
        {"user": "Aisha", "amount": None},
        {"user": "Kumar", "amount": 800},
        {"user": "Ravi", "amount": 500},
        {"user": "John", "amount": -100}
    ]

In [18]:
def clean(data):
  logging.info("Cleaning step")
  return [d for d in data if d['amount'] is not None and d['amount']>0]

In [19]:
def transform(data):
  logging.info("Transform Step")
  for d in data:
    d['amount_with_tax']=d['amount']*1.1
  return data

In [20]:
def load(data):
  logging.info('Load step')
  result={}
  for d in data:
    user = d['user']
    result[user] = result.get(user,0)+d['amount_with_tax']
  print("Final Aggregated Data:", result)

In [21]:
dag = {
    "extract": {"func": extract, "next": ["clean"]},
    "clean": {"func": clean, "next": ["transform"]},
    "transform": {"func": transform, "next": ["load"]},
    "load": {"func": load, "next": []}
}

In [22]:
def execute_with_retry(func, input_data=None, retries=2):
    for attempt in range(retries):
        try:
            if input_data is not None:
                return func(input_data)
            else:
                return func()
        except Exception as e:
            logging.error(f"Error: {e}, Retry {attempt+1}")
            time.sleep(1)
    raise Exception("Task failed after retries")

In [23]:
def run_dag():
    data_store = {}

    order = ["extract", "clean", "transform", "load"]

    for task in order:
        func = dag[task]["func"]

        if task == "extract":
            output = execute_with_retry(func)
        else:
            prev_task = order[order.index(task) - 1]
            output = execute_with_retry(func, data_store[prev_task])

        data_store[task] = output

    logging.info("DAG Execution Completed")

In [24]:
run_dag()

2026-03-29 05:27:55,557 - INFO - Extract step
2026-03-29 05:27:55,558 - INFO - Cleaning step
2026-03-29 05:27:55,559 - INFO - Transform Step
2026-03-29 05:27:55,559 - INFO - Load step
Final Aggregated Data: {'Ravi': 1870.0, 'Kumar': 880.0000000000001}
2026-03-29 05:27:55,560 - INFO - DAG Execution Completed
